<a href="https://colab.research.google.com/github/kamalkalyan13/Pyhton-baisc-assignment/blob/main/Module_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Module 1

Task 1 — Scrape books


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


BASE_URL = "https://books.toscrape.com/catalogue/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"


def scrape_books():
    books = []
    url = START_URL

    for page in range(1, 6):
        response = requests.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for book in soup.select("article.product_pod"):
            title = book.h3.a["title"]

            price = book.select_one(".price_color").text.strip()

            rating = book.select_one(".star-rating")["class"][1]

            availability = book.select_one(".availability").text.strip()

            relative_link = book.h3.a["href"]

            book_url = requests.compat.urljoin(url, relative_link)

            category_response = requests.get(book_url)
            category_response.raise_for_status()

            category_soup = BeautifulSoup(
                category_response.text,
                "html.parser"
            )

            breadcrumb = category_soup.select("ul.breadcrumb li a")

            if len(breadcrumb) >= 3:
                category = breadcrumb[-1].text.strip()
            else:
                category = "Unknown"

            books.append({
                "title": title,
                "price": price,
                "star_rating": rating,
                "availability": availability,
                "category": category
            })

        next_page = soup.select_one("li.next a")

        if next_page:
            url = requests.compat.urljoin(url, next_page["href"])
        else:
            break

    return pd.DataFrame(books)


df = scrape_books()

print("Number of books:", len(df))
print("Number of categories:", df["category"].nunique())
print(df.head())

df.to_csv("raw_books.csv", index=False)

Number of books: 100
Number of categories: 29
                                   title    price star_rating availability  \
0                   A Light in the Attic  Â£51.77       Three     In stock   
1                     Tipping the Velvet  Â£53.74         One     In stock   
2                             Soumission  Â£50.10         One     In stock   
3                          Sharp Objects  Â£47.82        Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23        Five     In stock   

             category  
0              Poetry  
1  Historical Fiction  
2             Fiction  
3             Mystery  
4             History  


Task 2 — Clean the data


In [5]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# Clean price
df["price_gbp"] = (
    df["price"]
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

# Convert rating text to numbers
df["rating"] = df["star_rating"].map(rating_map)

# Convert availability to True/False
df["in_stock"] = (
    df["availability"]
    .str.contains("In stock", case=False, na=False)
)

# Handle missing numeric values
df["price_gbp"] = df["price_gbp"].fillna(
    df["price_gbp"].median()
)

df["rating"] = df["rating"].fillna(
    df["rating"].median()
).astype(int)

# Remove rows with essential missing values
df = df.dropna(
    subset=["title", "category", "in_stock"]
)

(df.head())
(df.dtypes)

,0
title,object
price,object
star_rating,object
availability,object
category,object
price_gbp,float64
rating,int64
in_stock,bool


Task 3 — Convert GBP to INR

In [7]:
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

print(df[[
    "title",
    "price_gbp",
    "price_inr"
]].head())

                                   title  price_gbp  price_inr
0                   A Light in the Attic      51.77   5461.735
1                     Tipping the Velvet      53.74   5669.570
2                             Soumission      50.10   5285.550
3                          Sharp Objects      47.82   5045.010
4  Sapiens: A Brief History of Humankind      54.23   5721.265


Task 4 — Create normalized SQLite database

In [14]:
import sqlite3

connection = sqlite3.connect("zepto_books.db")
cursor = connection.cursor()

# Create categories table
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

# Create books table
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

connection.commit()

print("Tables created successfully")
categories = df["category"].dropna().unique()

for category in categories:
    cursor.execute(
        """
        INSERT OR IGNORE INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )

connection.commit()

print("Categories inserted successfully")
for _, row in df.iterrows():

    cursor.execute(
        """
        SELECT category_id
        FROM categories
        WHERE category_name = ?
        """,
        (row["category"],)
    )

    category_id = cursor.fetchone()[0]

    cursor.execute(
        """
        INSERT INTO books
        (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["rating"],
            int(row["in_stock"]),
            category_id
        )
    )

connection.commit()

print("Books inserted successfully")
print(
    "Books:",
    cursor.execute("SELECT COUNT(*) FROM books").fetchone()[0]
)

print(
    "Categories:",
    cursor.execute("SELECT COUNT(*) FROM categories").fetchone()[0]
)
connection.close()

Tables created successfully
Categories inserted successfully
Books inserted successfully
Books: 100
Categories: 29


In [17]:
query1 = """
SELECT title, price_gbp
FROM books
WHERE price_gbp > 40;
"""
result1 = pd.read_sql(query1, connection)

print("\nQUERY 1")
print(query1)
print(result1)

query2 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC;
"""

result2 = pd.read_sql(query2, connection)

print("\nQUERY 2")
print(query2)
print(result2)

query3 = """
SELECT title, rating
FROM books
ORDER BY rating DESC
LIMIT 10;
"""

result3 = pd.read_sql(query3, connection)

print("\nQUERY 3")
print(query3)
print(result3)

query4 = """
SELECT DISTINCT category_name
FROM categories
WHERE category_name IN ('Poetry', 'Mystery', 'Fiction');
"""

result4 = pd.read_sql(query4, connection)

print("\nQUERY 4")
print(query4)
print(result4)

query5 = """
SELECT
    b.title,
    b.rating,
    b.price_gbp,
    c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
ORDER BY b.rating DESC
LIMIT 10;
"""

result5 = pd.read_sql(query5, connection)

print("\nQUERY 5")
print(query5)
print(result5)
connection.close()

ProgrammingError: Cannot operate on a closed database.

Task 6 — pd.read_sql() and pd.merge()

In [21]:
import sqlite3
import pandas as pd


connection = sqlite3.connect("zepto_books.db")
cursor = connection.cursor()

cursor.execute("PRAGMA foreign_keys = ON")

cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")


cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")


cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")



categories = df["category"].dropna().unique()

for category in categories:
    cursor.execute(
        """
        INSERT INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )



for _, row in df.iterrows():

    cursor.execute(
        """
        SELECT category_id
        FROM categories
        WHERE category_name = ?
        """,
        (row["category"],)
    )

    category_id = cursor.fetchone()[0]

    cursor.execute(
        """
        INSERT INTO books
        (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["rating"],
            int(row["in_stock"]),
            category_id
        )
    )


connection.commit()

print("Database created successfully.")
print("Data inserted successfully.")


book_count = cursor.execute(
    "SELECT COUNT(*) FROM books"
).fetchone()[0]

category_count = cursor.execute(
    "SELECT COUNT(*) FROM categories"
).fetchone()[0]

print("Number of books:", book_count)
print("Number of categories:", category_count)


query1 = """
SELECT title, price_gbp
FROM books
WHERE price_gbp > 40;
"""

result1 = pd.read_sql(query1, connection)

print("\n========== QUERY 1 ==========")
print(query1)
print(result1)


query2 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC;
"""

result2 = pd.read_sql(query2, connection)

print("\n========== QUERY 2 ==========")
print(query2)
print(result2)


query3 = """
SELECT title, rating
FROM books
ORDER BY rating DESC
LIMIT 10;
"""

result3 = pd.read_sql(query3, connection)

print("\n========== QUERY 3 ==========")
print(query3)
print(result3)




query4 = """
SELECT DISTINCT category_name
FROM categories
WHERE category_name IN ('Poetry', 'Mystery', 'Fiction');
"""

result4 = pd.read_sql(query4, connection)

print("\n========== QUERY 4 ==========")
print(query4)
print(result4)



query5 = """
SELECT
    b.title,
    b.rating,
    b.price_gbp,
    c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
ORDER BY b.rating DESC
LIMIT 10;
"""

result5 = pd.read_sql(query5, connection)

print("\n========== QUERY 5 ==========")
print(query5)
print(result5)



books_df = pd.read_sql(
    "SELECT * FROM books",
    connection
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    connection
)

print("\n========== BOOKS DATAFRAME ==========")
print(books_df.head())

print("\n========== CATEGORIES DATAFRAME ==========")
print(categories_df.head())


merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)


# Select the same columns as SQL query
merge_result = merge_result[
    [
        "title",
        "rating",
        "price_gbp",
        "category_name"
    ]
]


merge_result = (
    merge_result
    .sort_values("rating", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

sql_result = result5.reset_index(drop=True)


print("\n========== SQL JOIN RESULT ==========")
print(sql_result)


print("\n========== PANDAS MERGE RESULT ==========")
print(merge_result)



print(
    "\nDo SQL JOIN and pandas merge match?",
    sql_result.equals(merge_result)
)


connection.close()

print("\nDatabase connection closed.")

Database created successfully.
Data inserted successfully.
Number of books: 100
Number of categories: 29

========== QUERY 1 ==========

SELECT title, price_gbp
FROM books
WHERE price_gbp > 40;

                                                title  price_gbp
0                                A Light in the Attic      51.77
1                                  Tipping the Velvet      53.74
2                                          Soumission      50.10
3                                       Sharp Objects      47.82
4               Sapiens: A Brief History of Humankind      54.23
5                                     The Black Maria      52.15
6   Scott Pilgrim's Precious Little Life (Scott Pi...      52.29
7   Our Band Could Be Your Life: Scenes from the A...      57.25
8                        Libertarianism for Beginners      51.33
9                             It's Only the Himalayas      45.17
10                      Birdsong: A Story in Pictures      54.64
11                     Al